<a href="https://colab.research.google.com/github/rajivagarwal221/Aahar-Kitchen-System/blob/train-detector/notebooks/train_aahar_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AAhar food detector — Run All

**Before pressing Run All, just two things:**

1. **Runtime → Change runtime type → T4 GPU**
2. **Left sidebar → 🔑 (Secrets)**, add three secrets with Notebook access ON:
   - `HF_TOKEN` — your Hugging Face write token (https://huggingface.co/settings/tokens)
   - `KAGGLE_USERNAME` — your Kaggle username
   - `KAGGLE_KEY` — your Kaggle API key (https://www.kaggle.com/settings → Create New Token)

Then **Runtime → Run all**. Walk away for 2–6 hours.

The notebook fetches the three Kaggle image datasets → normalizes folder-per-class into YOLO format → fine-tunes yolo11m on T4 → exports ONNX + TFLite → pushes to `rajivagarwal221/aahar-detector` on Hugging Face.

## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.* huggingface_hub==0.26.* kaggle onnx onnxruntime 2>&1 | tail -3
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Config

In [ ]:
HF_MODEL_REPO = 'rajivagarwal221/aahar-detector'   # change to use a different HF account

KAGGLE_DATASETS = [
    'iamsouravbanerjee/indian-food-images-dataset',     # 80 dish classes
    'kritikseth/fruit-and-vegetable-image-recognition', # 33 produce
    'tankitrathaur/indian-spices-images-data',          # Indian spices
]

MODEL     = 'yolo11m.pt'   # m = ~20M params, fits T4; use yolo11s.pt if OOM
IMG_SIZE  = 640
EPOCHS    = 80
BATCH     = 16
VAL_SPLIT = 0.1

## 3. Credentials — reads Colab secrets

In [ ]:
import os, json
from pathlib import Path
from google.colab import userdata

HF_TOKEN        = userdata.get('HF_TOKEN')
KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY      = userdata.get('KAGGLE_KEY')

assert HF_TOKEN,        'Missing HF_TOKEN in Colab secrets'
assert KAGGLE_USERNAME, 'Missing KAGGLE_USERNAME in Colab secrets'
assert KAGGLE_KEY,      'Missing KAGGLE_KEY in Colab secrets'

kcfg = Path.home() / '.kaggle'
kcfg.mkdir(exist_ok=True)
(kcfg / 'kaggle.json').write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
(kcfg / 'kaggle.json').chmod(0o600)
print('Kaggle + HF credentials ready')

## 4. Fetch image datasets from Kaggle (~4 GB)

In [ ]:
import subprocess
RAW = Path('/content/raw_dataset')
RAW.mkdir(exist_ok=True)

for ref in KAGGLE_DATASETS:
    out = RAW / ref.split('/')[-1]
    out.mkdir(exist_ok=True)
    print(f'↓ {ref}')
    subprocess.run(['kaggle','datasets','download','-d',ref,'-p',str(out),'--unzip'], check=True)

for ref in KAGGLE_DATASETS:
    root = RAW / ref.split('/')[-1]
    print('\n', root)
    for p in sorted(root.iterdir())[:3]:
        print(' ', p.name, '→', len(list(p.iterdir())) if p.is_dir() else 'file')

## 5. Normalize folder-per-class → YOLO layout

Each image gets a full-frame bounding box (class 0..N–1 from its folder name). Good enough for food-ID — the detector learns to localize and classify dishes. Swap in real bboxes later for tighter crops.

In [ ]:
import shutil, random, yaml
from PIL import Image

DATA = Path('/content/yolo_data')
for sub in ('images/train','images/val','labels/train','labels/val'):
    (DATA / sub).mkdir(parents=True, exist_ok=True)

IMG_EXTS = {'.jpg','.jpeg','.png','.webp'}

class_dirs = {}
for base in RAW.iterdir():
    if not base.is_dir(): continue
    for p in base.rglob('*'):
        if p.is_dir() and any(c.suffix.lower() in IMG_EXTS for c in p.iterdir() if c.is_file()):
            name = p.name.strip().lower().replace('_',' ')
            class_dirs.setdefault(name, []).append(p)

class_names = sorted(class_dirs.keys())
print(f'{len(class_names)} unique classes discovered')

random.seed(42)
n_train = n_val = 0
for cid, name in enumerate(class_names):
    imgs = []
    for d in class_dirs[name]:
        imgs.extend(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
    random.shuffle(imgs)
    n_v = max(1, int(len(imgs) * VAL_SPLIT)) if len(imgs) > 1 else 0
    for i, img in enumerate(imgs):
        try:
            with Image.open(img) as im: im.verify()
        except Exception:
            continue
        split = 'val' if i < n_v else 'train'
        dst_img = DATA / f'images/{split}' / f'{cid:04d}_{img.stem}_{i}{img.suffix.lower()}'
        dst_lbl = DATA / f'labels/{split}' / f'{cid:04d}_{img.stem}_{i}.txt'
        shutil.copy2(img, dst_img)
        dst_lbl.write_text(f'{cid} 0.5 0.5 1.0 1.0\n')
        if split == 'val': n_val += 1
        else: n_train += 1

cfg_path = Path('/content/aahar.yaml')
cfg_path.write_text(yaml.safe_dump({
    'path': str(DATA),
    'train': 'images/train',
    'val':   'images/val',
    'names': class_names,
}, sort_keys=False))
print(f'train={n_train} val={n_val} classes={len(class_names)}')

## 6. Train

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=str(cfg_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    project='/content/runs',
    name='aahar',
    patience=20,
    plots=True,
)
best = Path(results.save_dir) / 'weights' / 'best.pt'
print('best:', best, 'exists:', best.exists())

## 7. Export ONNX + TFLite

In [ ]:
trained = YOLO(str(best))
onnx_path   = trained.export(format='onnx',   imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True)
tflite_path = trained.export(format='tflite', imgsz=IMG_SIZE, int8=False)
print('onnx:',   onnx_path)
print('tflite:', tflite_path)

## 8. Push to Hugging Face

In [ ]:
from huggingface_hub import HfApi, create_repo
api = HfApi(token=HF_TOKEN)
create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)

for f in [best, Path(onnx_path), Path(tflite_path), cfg_path]:
    api.upload_file(path_or_fileobj=str(f), path_in_repo=f.name, repo_id=HF_MODEL_REPO, repo_type='model')
    print('uploaded', f.name)

print(f'\n→ https://huggingface.co/{HF_MODEL_REPO}')